# Shuffle Skew and Salting

Companion notebook for the [Partitioning, Skew and Salting](https://codemia.io/courses/big_data_engines/bd_partitioning_skew_and_salting) lesson on Codemia.

**What you will build**: a shuffle you can measure. You will build a realistic key distribution, hash partition it the way
Spark does, find the straggler task that decides when the stage ends, then salt the hot keys and
watch the stage time collapse. The sweep at the end shows where the salt stops helping and starts
costing.

## Setup

In [ ]:
%pip install -q numpy matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(7)

## Concept recap

A shuffle sends every record to the partition its key hashes to, and a stage ends when its slowest
task ends. Those two sentences are the whole lesson: if one partition holds ten times the mean, the
stage takes ten times longer than the cluster's capacity says it should, and no amount of extra
executors fixes it, because the work is not divisible by the machines you added.

The distribution below is the one you actually get in production. Most keys are small, a long tail
of keys is medium, and a handful of keys belong to the three customers your sales team is proudest
of.

## The key distribution

In [ ]:
N_EVENTS = 2_000_000
N_KEYS = 50_000

# A Zipf-like popularity law, plus three accounts far larger than the law predicts.
ranks = np.arange(1, N_KEYS + 1)
weights = 1.0 / ranks ** 0.8
weights[:3] *= 4
weights /= weights.sum()

keys = rng.choice(N_KEYS, size=N_EVENTS, p=weights)
counts = np.bincount(keys, minlength=N_KEYS)
order = np.argsort(counts)[::-1]

print(f'events          {N_EVENTS:>12,}')
print(f'distinct keys   {len(np.unique(keys)):>12,}')
print(f'mean per key    {counts.mean():>12,.1f}')
print(f'largest key     {counts[order[0]]:>12,}')
print()
print('rank   records     share of all events')
for r in range(5):
    c = counts[order[r]]
    print(f'{r + 1:>4}{c:>10,}{c / N_EVENTS:>19.2%}')

top_1pct = counts[order[: N_KEYS // 100]].sum()
print()
print(f'top 1% of keys carry {top_1pct / N_EVENTS:.1%} of the events')

Nothing here is pathological. This is a normal customer table, and it already has the property that
breaks a shuffle: a small set of keys owns a large share of the rows.

## Hash partitioning, and the task that decides the stage time

In [ ]:
PARTITIONS = 200
RECORDS_PER_SECOND = 400_000       # what one task can chew through


def hash_partition(key_array, partitions, salt=None):
    """Knuth multiplicative hash, which spreads about as well as murmur3 for integer keys."""
    h = key_array.astype(np.int64) * 2654435761
    if salt is not None:
        h = h + salt.astype(np.int64) * 40503
    return (h % (2 ** 32)) % partitions


def task_report(sizes, label):
    seconds = sizes / RECORDS_PER_SECOND
    ideal = sizes.sum() / len(sizes) / RECORDS_PER_SECOND
    print(f'{label}')
    print(f'  largest partition {sizes.max():>10,} records   {seconds.max():>7.2f} s')
    print(f'  median partition  {int(np.median(sizes)):>10,} records   {np.median(seconds):>7.2f} s')
    print(f'  skew ratio        {sizes.max() / np.median(sizes):>10.1f} x median')
    print(f'  stage wall clock  {seconds.max():>10.2f} s   (perfectly balanced: {ideal:.2f} s)')
    print(f'  wasted capacity   {1 - seconds.mean() / seconds.max():>10.1%} of the slot time')
    return seconds.max()


parts = hash_partition(keys, PARTITIONS)
sizes = np.bincount(parts, minlength=PARTITIONS)
plain_stage = task_report(sizes, f'{PARTITIONS} partitions, plain hash partitioning')

The skew ratio is the number to carry around. A stage whose largest partition is several times its
median is a stage that is mostly idle: every other task finished long ago and their slots are being
held open while one task works. The wasted capacity line is what that costs you, and it is the
money you are paying for a cluster that is not running.

In [ ]:
plt.figure(figsize=(10, 4))
plt.bar(range(PARTITIONS), np.sort(sizes)[::-1], width=1.0, color='tab:red')
plt.axhline(sizes.sum() / PARTITIONS, color='black', linestyle='--',
            label=f'perfectly balanced ({sizes.sum() // PARTITIONS:,} records)')
plt.xlabel('partitions, largest first')
plt.ylabel('records in the partition')
plt.title('One shuffle, 200 partitions: the first few bars are the whole stage')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

Raising the partition count does not flatten those first bars. A single key cannot be split by
hashing, so its partition is a floor that more partitions cannot go below. Try it and watch the
median fall while the maximum does not.

In [ ]:
print(f"{'partitions':>12}{'largest':>12}{'median':>10}{'skew':>8}{'stage s':>10}")
for p in [200, 400, 1000, 4000]:
    s = np.bincount(hash_partition(keys, p), minlength=p)
    print(f'{p:>12}{s.max():>12,}{int(np.median(s)):>10,}'
          f'{s.max() / np.median(s):>8.1f}{s.max() / RECORDS_PER_SECOND:>10.2f}')

Twenty times the partitions, and the stage still waits for the same task. This is the experiment
worth remembering the next time somebody proposes fixing a slow job by repartitioning it harder.

## Salting the hot keys

In [ ]:
def salted_sizes(keys, partitions, salt_factor, hot_keys):
    """Split the chosen hot keys across salt_factor sub-keys; leave everything else alone."""
    is_hot = np.isin(keys, hot_keys)
    salt = np.zeros(len(keys), dtype=np.int64)
    salt[is_hot] = rng.integers(0, salt_factor, size=is_hot.sum())
    parts = hash_partition(keys, partitions, salt=salt)
    return np.bincount(parts, minlength=partitions), is_hot.sum()


HOT = 20                    # how many keys we treat as hot
SALT = 16                   # how many sub-keys each hot key becomes

hot_keys = order[:HOT]
print(f'salting the {HOT} largest keys, which carry '
      f'{counts[hot_keys].sum() / N_EVENTS:.1%} of all events, into {SALT} sub-keys each')
print()

salted, hot_rows = salted_sizes(keys, PARTITIONS, SALT, hot_keys)
salted_stage = task_report(salted, f'{PARTITIONS} partitions, {HOT} keys salted {SALT} ways')
print()
print(f'stage time {plain_stage:.2f} s -> {salted_stage:.2f} s '
      f'({plain_stage / salted_stage:.1f}x faster, same cluster)')

The stage got faster without a single extra executor. That is the point of salting: it is not a
performance trick, it is the removal of an artificial dependency between one key and one task.

In [ ]:
plt.figure(figsize=(10, 4))
plt.bar(range(PARTITIONS), np.sort(sizes)[::-1], width=1.0, color='tab:red', label='plain hash')
plt.bar(range(PARTITIONS), np.sort(salted)[::-1], width=1.0, color='tab:blue', alpha=0.8,
        label=f'{HOT} keys salted {SALT} ways')
plt.axhline(sizes.sum() / PARTITIONS, color='black', linestyle='--', label='perfectly balanced')
plt.xlabel('partitions, largest first')
plt.ylabel('records in the partition')
plt.title('Salting removes the floor that hashing put under the hot keys')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## How much salt, and what it costs

In [ ]:
print(f"{'salt':>6}{'largest':>12}{'skew':>8}{'stage s':>10}{'partial rows':>15}{'extra pass':>12}")
sweep = []
for salt_factor in [1, 2, 4, 8, 16, 32, 64, 128]:
    s, _ = salted_sizes(keys, PARTITIONS, salt_factor, hot_keys)
    partial_rows = HOT * salt_factor            # what the second aggregation has to re-combine
    stage = s.max() / RECORDS_PER_SECOND
    sweep.append((salt_factor, stage, s.max() / np.median(s)))
    print(f'{salt_factor:>6}{s.max():>12,}{s.max() / np.median(s):>8.1f}{stage:>10.2f}'
          f'{partial_rows:>15,}{partial_rows / RECORDS_PER_SECOND:>11.3f}s')

In [ ]:
factors = [f for f, _, _ in sweep]
stages = [s for _, s, _ in sweep]

plt.figure(figsize=(9, 4.5))
plt.plot(factors, stages, marker='o', color='tab:blue')
plt.axhline(sizes.sum() / PARTITIONS / RECORDS_PER_SECOND, color='black', linestyle='--',
            label='perfectly balanced stage')
plt.xscale('log', base=2)
plt.xlabel('salt factor (sub-keys per hot key)')
plt.ylabel('stage wall clock (seconds)')
plt.title('Salt until the hot keys stop being the largest partition, then stop')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

The curve flattens, and it flattens for a reason worth stating: once a hot key's share is below the
mean partition size, the straggler is no longer that key, it is ordinary variance in the hash. Salt
past that point and you are paying for partial rows and a second aggregation pass that buys nothing.

The partial row column is the honest cost side. Every salted group has to be re-combined, so the
query grows a stage. For an aggregation that is cheap, because the partials are tiny compared with
the input. For a join it is worse: the small side has to be exploded by the salt factor so that
every sub-key can find its match, which multiplies the broadcast or the build side.

## What the engine does for you, and what it does not

In [ ]:
# Adaptive execution splits oversized partitions after the fact, which needs no key knowledge.
TARGET = int(sizes.sum() / PARTITIONS * 2)          # Spark's skewed-partition threshold, roughly

def aqe_split(sizes, target):
    """Each oversized partition becomes ceil(size / target) tasks of roughly equal size."""
    out = []
    for s in sizes:
        if s <= target:
            out.append(s)
        else:
            pieces = -(-s // target)
            out.extend([s // pieces] * pieces)
    return np.array(out)


aqe = aqe_split(sizes, TARGET)
print(f'threshold {TARGET:,} records per partition')
print(f'tasks {len(sizes)} -> {len(aqe)}')
print(f'largest task {sizes.max():,} -> {aqe.max():,}')
print(f'stage wall clock {sizes.max() / RECORDS_PER_SECOND:.2f} s -> '
      f'{aqe.max() / RECORDS_PER_SECOND:.2f} s')

Splitting after the fact gets most of the win with none of the query rewriting, which is why
adaptive execution is on by default in modern Spark and why hand written salting is rarer than it
was. It has one hard limit: a split can only divide a partition, and a partition that is one key is
indivisible on the probe side of a join without duplicating the other side. That is the case where
you still reach for salt, and the case where the only real fix is to change the key.

## What to take away

- A stage costs what its largest task costs. Averages describe a cluster that is not the one you
  are running.
- More partitions lowers the median and leaves the maximum where it was, because hashing cannot
  split a key.
- Salting lowers the maximum, costs an extra pass over the partials, and stops helping as soon as
  the hot key is no longer the largest partition. The sweep tells you where that is.
- Adaptive skew handling does the same thing without your help, up to the point where the oversized
  partition is a single key.
- Before any of that, check whether the hot key is real. Half the skew in production is a null, an
  empty string or a sentinel like `-1`, and the fix is a filter rather than a strategy.